In [0]:
from pyspark.sql import SparkSession

# Create or get existing Spark session
spark = SparkSession.builder.getOrCreate()

import pandas as pd

# Python 'open' can always see the Repo, even when Spark is restricted
local_path = "/Workspace/Users/sahityagantalausa@gmail.com/data_engineering_projects/databricks/legacy_fuel_data.csv"
pdf = pd.read_csv(local_path)

# Convert the local Pandas object into a distributed Spark DataFrame
df_csv = spark.createDataFrame(pdf)

In [0]:
#pdf = spark.read.format("csv").options(header="True",inferSchema="True").load(local_path)
# TASK 3: Data quality checks (10 min)
# 1. Check for nulls
# 2. Validate fuel_consumed > 0
# 3. Calculate fuel efficiency (km per liter)
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum, when ,isnull

df_scv =df_csv.select([sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in df_csv.columns])
df_scv = df_csv.filter("fuel_consumed > 0")
df_scv = df_csv.withColumn("fuel_efficiency",df_csv["distance_km"]/df_csv["fuel_consumed"])
df_scv.show()
# DBTITLE 1,")

In [0]:
# TASK 3: Data Quality Checks (10 min)

# DBTITLE 1,Step 1: Identify Null Values Across All Columns
# We aggregate the count of nulls for every column in the dataset
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum, when, isnull

# Note: Corrected variable name from df_scv to df_csv for consistency
null_counts_df = df_csv.select([
    sum(when(isnull(c), 1).otherwise(0)).alias(c) 
    for c in df_csv.columns
])
null_counts_df.show()

# DBTITLE 1,Step 2: Filter Valid Records (Fuel Consumed > 0)
# This validates that we aren't processing impossible data (0 or negative fuel)
df_valid_fuel = df_csv.filter("fuel_consumed > 0")

# DBTITLE 1,Step 3: Calculate Fuel Efficiency (km per liter)
# We create a new derived column for performance analysis
df_final = df_valid_fuel.withColumn(
    "fuel_efficiency", 
    col("distance_km") / col("fuel_consumed")
)

# Display the final results for Task 3
df_final.show()

In [0]:
# TASK 4: Write to Delta with partitioning (10 min)
# Partition by train_id for query optimization
# YOUR CODE HERE:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("train_id") \
    .saveAsTable("silver_fuel_efficiency")

In [0]:
# 1. Get the actual location of the managed table
table_info = spark.sql("DESCRIBE DETAIL silver_fuel_efficiency")

# 3. Read using table name instead of path
df_delta = spark.read.table("silver_fuel_efficiency")
print(f"Delta table records: {df_delta.count()}")


## Phase 1: Discovery & Schema Contract
- "Data Contract" between the legacy system and the new platform
- inventory & Metadata Cataloging: Catalog every CSV source, owner, update frequency, and expected row count.
- Data Profiling: Run profiling scripts to detect anomalies in the legacy data before migration (e.g., hidden NULLs, mixed date formats like MM/DD/YY vs YYYY-MM-DD, and "garbage" characters in string fields).
- Encoding Audit: CSVs are notorious for encoding issues. Explicitly identify if files are UTF-8, Latin-1, or Windows-1252 to prevent character corruption during ingestion.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Define the Strict Data Contract (DDL)
# This prevents "Schema Drift" from breaking downstream ML models
train_schema = StructType([
    StructField("train_id", StringType(), False),
    StructField("route", StringType(), True),
    StructField("distance_miles", DoubleType(), True),
    StructField("weather_condition", StringType(), True),
    StructField("cargo_weight_tons", DoubleType(), True),
    StructField("scheduled_duration_hours", DoubleType(), True),
    StructField("actual_duration_hours", DoubleType(), True),
    StructField("crew_experience_years", IntegerType(), True),
    StructField("locomotive_age_years", IntegerType(), True),
    StructField("maintenance_score", IntegerType(), True),
    StructField("delayed", IntegerType(), True)
])

## Phase 2.Strategic Ingestion Design

-  you should avoid simple spark.read.csv. Use modern ingestion patterns:
- When data comes from:
    API response,
    Web request,
    Database query,
    Spark .collect(),
    Generated dynamically in code,
    When you don’t have a physical file -- then use 
- spark handles this differently parallize and utilize 
- Spark:

    -- Converts input into partitions
    -- Distributes partitions across executors
    -- Processes in parallel
    -- Lazily evaluates transformations
    -- There is no “file open” like normal Python.
- Instead:
    -- Data source → RDD → DataFrame → Catalyst optimizer → Execution
- Batch Processing - Data already exists and is complete. [check here](https://dbc-63148d35-2fa5.cloud.databricks.com/editor/files/1333808587609777?o=1861436472604188)
    - CSV file in blob storage
    - Table in database
    - Parquet file in S3
    - Historical dataset
- stream processing - Data keeps arriving continuously.[check here](https://dbc-63148d35-2fa5.cloud.databricks.com/editor/files/1333808587609778?o=1861436472604188)
    - Kafka topic
    - Event stream
    - IoT sensor data
    - Live application logs
- Auto Loader (cloudFiles): Use this for incremental migration. It manages "State" (which files were processed) and handles Schema Evolution automatically. 
- Schema Hints: If your CSVs lack headers or have ambiguous types (e.g., a "Zip Code" column that looks like an integer but should be a string), define a Schema Hint or explicit DDL to prevent Spark from dropping leading zeros.
- Quarantine Strategy (Bad Record Path): Configure badRecordsPath or use Databricks Expectations to catch rows that fail parsing. Instead of the whole job failing, the "bad" rows are shunted to a side-table for manual review.
- Some rows may:
    - Have wrong data type
    - Missing required column
    - Corrupt format
    - Negative delay
    - Invalid timestamp
- Bronze Strategy -- quarentine implementation -- load everthing -- add as metadata -- move bad records to quarentine 
- for this .option("mode", "PERMISSIVE")
- .option("columnNameOfCorruptRecord", "_corrupt_record")
- dropping .option("mode", "DROPMALFORMED") -- removes bad data
- .option("mode", "FAILFAST") -- fail process 

In [0]:
import pandas as pd

pdf = pd.read_csv("")
df_bronze = spark.createDataFrame(pdf, schema=train_schema)

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# Read CSV with pandas
pdf = pd.read_csv("/Workspace/Users/sahityagantalausa@gmail.com/data_engineering_projects/databricks/train_delays.csv")

# Convert to Spark DataFrame
df_bronze = spark.createDataFrame(pdf, schema=train_schema)

# Add audit columns
df_bronze = df_bronze.withColumn("_ingested_at", F.current_timestamp()) \
                     .withColumn("_source_file", F.lit("train_delays.csv")) \
                     .withColumn("_batch_id", F.lit("manual_ingest"))  # pandas → Spark, no _metadata

df_bronze.printSchema()
display(df_bronze)

## Phase 3. The Medallion Transformation (Bronze to Silver)

- Convert those raw, messy CSVs into reliable Delta Tables.
- Deduplication: CSV migrations often involve re-processing files. Implement Idempotent logic(needed to be applied multiple times with out changing system state) using MERGE INTO or dropDuplicates() to ensure you don't double-count records.
    - dropduplicates() expensive on huge tables
    - merge into (delta_table.merge(bronze_new_table,<condition>).whenNotMatchedInsertAll() .execute() / matched)
- Audit Columns: Every record should be "stamped" with metadata: _ingestion_timestamp, _source_file_name, and _processing_batch_id. This is critical for 3:00 AM debugging.
    - above we can see df_broze auditing
- Type Casting & Normalization: Convert string-based dates to TIMESTAMP and currency strings to DECIMAL.

In [0]:
restricted_columns = [c for c in df_bronze.columns if "source" in c or "file" in c]
df_silver_base = df_bronze.drop(*restricted_columns)

# --- STEP 3: Deduplication & Quality Filtering (Silver Logic) ---

# Ensuring we don't have duplicate train logs for the same route/id
df_silver = df_silver_base.dropDuplicates(["train_id", "route", "scheduled_duration_hours"])

# Data Quality: Calculated field for duration delta
df_silver = df_silver.withColumn(
    "delay_minutes", 
    (F.col("actual_duration_hours") - F.col("scheduled_duration_hours")) * 60
)

# 1. Cast delay_minutes to integer
df_silver = df_silver.withColumn("delay_minutes", F.col("delay_minutes").cast("integer"))



# --- STEP 4: Write to Delta (Silver Table) ---

# This triggers the action. Because we dropped the input_file_name columns 
# in Step 0, the Unity Catalog error will be bypassed.
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.train_silver")

# Success check
print("Table 'train_silver' successfully updated in Delta format.")

## Phase 4. Schema Evolution & Governance
- Schema Enforcement: Switch from the "Wild West" of CSV to Delta’s Schema Enforcement. This ensures a source system change doesn't break your downstream dashboards.
- The Additive Pattern: If a new column is added to the CSV, configure your pipeline to allow the change (Expand), but prevent the removal of existing columns (Contract).
- Unity Catalog Integration: Tag columns with sensitive data (PII) using Attribute-Based Access Control (ABAC). This ensures that even though you migrated the data, only authorized users can see the "Social Security Number" column.

## Phase 5. Performance Optimization

- CSV is a "row-based" text format; Delta is a "columnar" binary format. You must optimize for this shift:
    - delta is better in mutiple performance ways and ,Enforced / evolution supported,Fast for filters, aggregates, joins
    - ACID, upserts, deletes
    - After ingesting CSV into Delta, you must optimize for analytics.
- Z-Ordering / Liquid Clustering: On your Silver/Gold tables, Z-Order by high-cardinality columns (like train_id or customer_id) to speed up filters.
    - spark.sql(""" OPTIMIZE train_silver ZORDER BY (train_id) """)
    - 
- File Compaction (OPTIMIZE): CSV ingestion often creates many small files. Use OPTIMIZE to bin-pack these into 1GB files to improve read performance by up to 100x.
- Vacuuming: Set a retention policy (e.g., 7 days) to clean up old versions of your Delta tables and save storage costs.#3

In [0]:
display(spark.sql(""" OPTIMIZE train_silver ZORDER BY (train_id) """))

In [0]:
spark.sql("VACUUM train_silver RETAIN 168 HOURS")  # keep 7 days